# MD Initial Structures with OpenMM + PACKMOL
Build, pack, and visualize step by step.

Goals:
1. Build one TIP3P water in OpenMM.
2. Solvate + ionize with `Modeller.addSolvent()`.
3. Compare DIY placement vs PACKMOL packing.
4. Reuse official PACKMOL **initial structures** but write readable `.inp` files in notebook cells.
5. Export all packed systems as PDB.
6. End with a fun `HELLO` water-art example.


## 0) Setup

This cell:
- Creates `outputs/`
- Installs OpenMM, py3Dmol, and PACKMOL on Colab if needed
- Downloads and extracts official PACKMOL examples archive

Archive URL:
`https://m3g.github.io/packmol/examples/examples.tar.gz`


In [58]:
import os
import sys
import shutil
import tarfile
import tempfile
import subprocess
import urllib.request
from pathlib import Path


def in_colab() -> bool:
    return "google.colab" in sys.modules


outdir = Path("outputs")
outdir.mkdir(exist_ok=True)

if in_colab():
    try:
        from google.colab import output  # type: ignore
        output.enable_custom_widget_manager()
    except Exception as exc:
        print("Colab widget manager error:", exc)

    def pip_install(pkg: str):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

    try:
        import openmm  # noqa: F401
    except Exception:
        pip_install("openmm")

    try:
        import py3Dmol  # noqa: F401
    except Exception:
        pip_install("py3Dmol")

    if shutil.which("packmol") is None:
        try:
            subprocess.check_call(["apt-get", "-qq", "update"])
            subprocess.check_call(["apt-get", "-qq", "install", "-y", "packmol"])
        except Exception as exc:
            print("Could not install packmol with apt-get:", exc)
            print("Try conda install -c conda-forge packmol")

PACKMOL_EXAMPLES_URL = "https://m3g.github.io/packmol/examples/examples.tar.gz"
packmol_examples_root = Path("packmol_examples")
packmol_examples_dir = packmol_examples_root / "examples"


def safe_extract_tar(tf: tarfile.TarFile, target_dir: Path):
    target_dir = target_dir.resolve()
    for member in tf.getmembers():
        out_path = (target_dir / member.name).resolve()
        if not str(out_path).startswith(str(target_dir)):
            raise RuntimeError(f"Unsafe tar member path: {member.name}")
    tf.extractall(target_dir)


if not packmol_examples_dir.exists():
    packmol_examples_root.mkdir(parents=True, exist_ok=True)
    fd, tmp_tar = tempfile.mkstemp(suffix=".tar.gz")
    os.close(fd)
    print("Downloading official PACKMOL examples...")
    urllib.request.urlretrieve(PACKMOL_EXAMPLES_URL, tmp_tar)
    with tarfile.open(tmp_tar, "r:gz") as tf:
        safe_extract_tar(tf, packmol_examples_root)
    os.remove(tmp_tar)

print("Running in Colab:", in_colab())
print("packmol found:", shutil.which("packmol") is not None)
print("outputs:", outdir.resolve())
print("examples:", packmol_examples_dir.resolve())


Running in Colab: True
packmol found: True
outputs: /content/outputs
examples: /content/packmol_examples/examples


## 1) Imports + helper functions

Small helpers for:
- topology summaries
- running PACKMOL from notebook text
- converting XYZ to PDB (used in interface example so final output is still PDB)


In [59]:
import json
import numpy as np
from pathlib import Path

from openmm import Vec3, unit
import openmm.app as app

try:
    import py3Dmol
    HAS_3DMOL = True
except Exception:
    HAS_3DMOL = False


def topology_summary(topology: app.Topology, title: str):
    n_atoms = sum(1 for _ in topology.atoms())
    n_res = sum(1 for _ in topology.residues())
    n_chains = sum(1 for _ in topology.chains())
    n_bonds = sum(1 for _ in topology.bonds())
    print(f"{title}: atoms={n_atoms}, residues={n_res}, chains={n_chains}, bonds={n_bonds}")


def random_rotation_matrix(rng: np.random.Generator):
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)],
    ])


def run_packmol(inp_text: str, inp_path: Path):
    inp_path.write_text(inp_text)
    print("Wrote PACKMOL input:", inp_path)

    if shutil.which("packmol") is None:
        print("packmol not found. Install and re-run this cell.")
        return 127

    # Some PACKMOL/Fortran builds require seekable stdin (not a pipe).
    with open(inp_path, "rb") as fin:
        proc = subprocess.run(
            ["packmol"],
            stdin=fin,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            cwd=inp_path.parent,
        )

    log = proc.stdout.decode("utf-8", errors="replace")
    print(log[:1800])
    return proc.returncode


def count_pdb_atoms(path: Path):
    n = 0
    for line in path.read_text().splitlines():
        if line.startswith("ATOM") or line.startswith("HETATM"):
            n += 1
    return n


def xyz_to_pdb(xyz_path: Path, pdb_path: Path, resname: str = "MOL"):
    lines = [line.strip() for line in xyz_path.read_text().splitlines() if line.strip()]
    n_atoms = int(lines[0])
    atom_lines = lines[2 : 2 + n_atoms]

    out = []
    for i, line in enumerate(atom_lines, start=1):
        parts = line.split()
        if len(parts) < 4:
            continue
        elem = parts[0].capitalize()
        x, y, z = map(float, parts[1:4])
        atom_name = elem if len(elem) <= 2 else elem[:2]
        out.append(
            f"HETATM{i:5d} {atom_name:>4s} {resname:>3s} A{1:4d}    {x:8.3f}{y:8.3f}{z:8.3f}{1.00:6.2f}{0.00:6.2f}          {elem:>2s}"
        )

    out.append("TER")
    out.append("END")
    pdb_path.write_text("\n".join(out) + "\n")


examples_dir = Path("packmol_examples") / "examples"
assert examples_dir.exists(), f"Missing examples directory: {examples_dir}"
print("py3Dmol installed:", HAS_3DMOL)
print("Using official templates from:", examples_dir.resolve())


py3Dmol installed: True
Using official templates from: /content/packmol_examples/examples


## 2) Build one TIP3P water (OpenMM)

No custom writer wrappers. We use OpenMM built-ins directly:
- `app.PDBFile.writeFile`
- `app.PDBxFile.writeFile`


In [60]:
r_oh = 0.09572 * unit.nanometer
angle_hoh = np.deg2rad(104.52)

top_single = app.Topology()
chain = top_single.addChain("A")
res = top_single.addResidue("HOH", chain)
O = top_single.addAtom("O", app.Element.getBySymbol("O"), res)
H1 = top_single.addAtom("H1", app.Element.getBySymbol("H"), res)
H2 = top_single.addAtom("H2", app.Element.getBySymbol("H"), res)
top_single.addBond(O, H1)
top_single.addBond(O, H2)

# Keep raw coordinates as Nx3 float array in nm, then wrap once with units.
single_xyz_nm = np.array([
    [0.0, 0.0, 0.0],
    [r_oh.value_in_unit(unit.nanometer), 0.0, 0.0],
    [
        r_oh.value_in_unit(unit.nanometer) * np.cos(angle_hoh),
        r_oh.value_in_unit(unit.nanometer) * np.sin(angle_hoh),
        0.0,
    ],
], dtype=float)
pos_single = single_xyz_nm * unit.nanometer

pdb_single = outdir / "01_single_tip3p.pdb"
cif_single = outdir / "01_single_tip3p.cif"

with open(pdb_single, "w") as f:
    app.PDBFile.writeFile(top_single, pos_single, f)

with open(cif_single, "w") as f:
    app.PDBxFile.writeFile(top_single, pos_single, f)

topology_summary(top_single, "Single TIP3P water")
print("Wrote:", pdb_single)
print("Wrote:", cif_single)


Single TIP3P water: atoms=3, residues=1, chains=1, bonds=2
Wrote: outputs/01_single_tip3p.pdb
Wrote: outputs/01_single_tip3p.cif


In [61]:
# Visualize single water
if not HAS_3DMOL:
    print("py3Dmol not installed.")
else:
    view = py3Dmol.view()
    view.addModel(open('outputs/01_single_tip3p.pdb', "r").read(), "pdb")
    view.setBackgroundColor("white")
    view.setStyle({"resn": "HOH"}, {"licorice": {"radius": 0.15, "colorscheme": "Jmol"}})
    view.zoomTo()
    view.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 3) Solvate + add ions with OpenMM

Build a 2.2 nm cubic water box at 0.15 M NaCl.


In [62]:
def make_forcefield():
    try:
        return app.ForceField("amber14-all.xml", "amber14/tip3p.xml")
    except Exception:
        return app.ForceField("amber99sbildn.xml", "tip3p.xml")


forcefield = make_forcefield()
modeller = app.Modeller(top_single, pos_single)

boxL = 2.2 * unit.nanometer
L_nm = boxL.value_in_unit(unit.nanometer)
box_size = Vec3(L_nm, L_nm, L_nm) * unit.nanometer

modeller.addSolvent(
    forcefield,
    model="tip3p",
    boxSize=box_size,
    ionicStrength=0.15 * unit.molar,
    neutralize=False,
    positiveIon="Na+",
    negativeIon="Cl-",
)

top_salt = modeller.topology
pos_salt = modeller.positions

pdb_salt = outdir / "02_waterbox_salt.pdb"
with open(pdb_salt, "w") as f:
    app.PDBFile.writeFile(top_salt, pos_salt, f)

topology_summary(top_salt, "Water + ions box")
print("Wrote:", pdb_salt)


Water + ions box: atoms=1016, residues=340, chains=3, bonds=676
Wrote: outputs/02_waterbox_salt.pdb


In [63]:
# Visualize solvated system
if not HAS_3DMOL:
    print("py3Dmol not installed.")
else:
    view = py3Dmol.view()
    view.addModel(open(pdb_salt, "r").read(), "pdb")
    view.setBackgroundColor("white")
    view.setStyle({"resn": "HOH"}, {"licorice": {"radius": 0.12, "colorscheme": "Jmol"}})
    view.addStyle({"resn": "NA"}, {"sphere": {"radius": 0.60, "color": "purple"}})
    view.addStyle({"resn": "CL"}, {"sphere": {"radius": 0.65, "color": "green"}})
    view.zoomTo()
    view.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 4) DIY packing (intuition)

Replicate one water on a grid with random rotations.


In [64]:
rng = np.random.default_rng(20260303)

template = single_xyz_nm.copy()

n_side = 7
spacing = L_nm / n_side

top_grid = app.Topology()
chain_g = top_grid.addChain("A")
positions_grid_nm = []

for ix in range(n_side):
    for iy in range(n_side):
        for iz in range(n_side):
            res = top_grid.addResidue("HOH", chain_g)
            o = top_grid.addAtom("O", app.Element.getBySymbol("O"), res)
            h1 = top_grid.addAtom("H1", app.Element.getBySymbol("H"), res)
            h2 = top_grid.addAtom("H2", app.Element.getBySymbol("H"), res)
            top_grid.addBond(o, h1)
            top_grid.addBond(o, h2)

            center = np.array([(ix + 0.5) * spacing, (iy + 0.5) * spacing, (iz + 0.5) * spacing])
            R = random_rotation_matrix(rng)
            coords = (template - template[0]) @ R.T + center
            for xyz in coords:
                positions_grid_nm.append(xyz)

positions_grid = np.asarray(positions_grid_nm, dtype=float) * unit.nanometer

# Topology box vectors are stored in nanometers; pass plain Vec3 values.
top_grid.setPeriodicBoxVectors((
    Vec3(L_nm, 0.0, 0.0),
    Vec3(0.0, L_nm, 0.0),
    Vec3(0.0, 0.0, L_nm),
))

pdb_grid = outdir / "03_waterbox_grid.pdb"
with open(pdb_grid, "w") as f:
    app.PDBFile.writeFile(top_grid, positions_grid, f)

topology_summary(top_grid, "DIY grid water")
print("waters:", n_side ** 3)
print("Wrote:", pdb_grid)


DIY grid water: atoms=1029, residues=343, chains=1, bonds=686
waters: 343
Wrote: outputs/03_waterbox_grid.pdb


In [65]:
# Visualize DIY grid result
if not HAS_3DMOL:
    print("py3Dmol not installed.")
else:
    view = py3Dmol.view()
    view.addModel(open(pdb_grid, "r").read(), "pdb")
    view.setBackgroundColor("white")
    view.setStyle({"resn": "HOH"}, {"licorice": {"radius": 0.12, "colorscheme": "Jmol"}})
    view.zoomTo()
    view.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 5) PACKMOL from scratch: same water box

We write the `.inp` text ourselves in this cell.


In [66]:
n_waters = n_side ** 3
box_A = 10.0 * L_nm

packmol_water_inp = outdir / "04_waterbox_from_scratch.inp"
packmol_water_pdb = outdir / "04_waterbox_packmol.pdb"

inp_water = (
    f"tolerance 2.0\n"
    f"filetype pdb\n"
    f"output {packmol_water_pdb.resolve()}\n\n"
    f"structure {pdb_single.resolve()}\n"
    f"  number {n_waters}\n"
    f"  inside box 0. 0. 0. {box_A:.3f} {box_A:.3f} {box_A:.3f}\n"
    f"end structure\n"
)

print("Notebook input for PACKMOL water box:\n")
print(inp_water)
_ = run_packmol(inp_water, packmol_water_inp)

if packmol_water_pdb.exists():
    top_pack = app.PDBFile(str(packmol_water_pdb)).topology
    topology_summary(top_pack, "PACKMOL water")
    print("Wrote:", packmol_water_pdb)


Notebook input for PACKMOL water box:

tolerance 2.0
filetype pdb
output /content/outputs/04_waterbox_packmol.pdb

structure /content/outputs/01_single_tip3p.pdb
  number 343
  inside box 0. 0. 0. 22.000 22.000 22.000
end structure

Wrote PACKMOL input: outputs/04_waterbox_from_scratch.inp

################################################################################

 PACKMOL - Packing optimization for the automated generation of
 starting configurations for molecular dynamics simulations.
 
                                                              Version 20.010 

################################################################################

  Packmol must be run with: packmol < inputfile.inp 

  Userguide at: http://m3g.iqm.unicamp.br/packmol 

  Reading input file... (Control-C aborts)
  Seed for random number generator:      1234567
  Output file: /content/outputs/04_waterbox_packmol.pdb
  Reading coordinate file: /content/outputs/01_single_tip3p.pdb
  Number of independ

In [67]:
# Visualize PACKMOL water result
if not HAS_3DMOL:
    print("py3Dmol not installed.")
elif not packmol_water_pdb.exists():
    print("Run previous cell first.")
else:
    view = py3Dmol.view()
    view.addModel(open(packmol_water_pdb, "r").read(), "pdb")
    view.setBackgroundColor("white")
    view.setStyle({"resn": "HOH"}, {"licorice": {"radius": 0.12, "colorscheme": "Jmol"}})
    view.zoomTo()
    view.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 6) Official PACKMOL initial structures + notebook `.inp` files

We use the downloaded official templates (`water.pdb`, `urea.pdb`, `protein.pdb`, etc.)
but we write readable `.inp` files in notebook cells so beginners can edit them.


## 6A) Simple mixture: water + urea


In [ ]:
print("Official mixture.inp:\n")
print((examples_dir / "mixture.inp").read_text())

mix_inp = outdir / "07_official_mixture.inp"
mix_out = outdir / "07_official_mixture.pdb"

inp_mix = (
    f"tolerance 2.0\n"
    f"filetype pdb\n"
    f"output {mix_out.resolve()}\n\n"
    f"structure {(examples_dir / 'water.pdb').resolve()}\n"
    f"  number 1000\n"
    f"  inside box 0. 0. 0. 40. 40. 40.\n"
    f"end structure\n\n"
    f"structure {(examples_dir / 'urea.pdb').resolve()}\n"
    f"  number 400\n"
    f"  inside box 0. 0. 0. 40. 40. 40.\n"
    f"end structure\n"
)

print("Notebook mixture input:\n")
print(inp_mix)
_ = run_packmol(inp_mix, mix_inp)

if mix_out.exists():
    topology_summary(app.PDBFile(str(mix_out)).topology, "Mixture output")
    print("Wrote:", mix_out)


In [ ]:
# Visualize mixture output
if not HAS_3DMOL:
    print("py3Dmol not installed.")
elif not mix_out.exists():
    print("Run previous cell first.")
else:
    view = py3Dmol.view()
    view.addModel(open(mix_out, "r").read(), "pdb")
    view.setBackgroundColor("white")
    view.setStyle({"resn": "HOH"}, {"licorice": {"radius": 0.12, "colorscheme": "Jmol"}})
    view.addStyle({"resn": "URE"}, {"stick": {"colorscheme": "yellowCarbon"}})
    view.zoomTo()
    view.show()


## 6B) Hormone at water/organic interface

Official templates are XYZ (`water.xyz`, `chlor.xyz`, `t3.xyz`).
For a clean beginner flow with PDB outputs, we first convert those template XYZ files to PDB,
then run PACKMOL in `filetype pdb` mode.


In [ ]:
print("Official interface.inp:\n")
print((examples_dir / "interface.inp").read_text())

iface_inp = outdir / "08_official_interface.inp"
iface_pdb = outdir / "08_official_interface.pdb"

# Convert official XYZ templates to PDB templates (one-time step for this tutorial)
water_xyz = examples_dir / "water.xyz"
chlor_xyz = examples_dir / "chlor.xyz"
t3_xyz = examples_dir / "t3.xyz"

water_pdb_tpl = outdir / "tpl_water_from_xyz.pdb"
chlor_pdb_tpl = outdir / "tpl_chlor_from_xyz.pdb"
t3_pdb_tpl = outdir / "tpl_t3_from_xyz.pdb"

xyz_to_pdb(water_xyz, water_pdb_tpl, resname="WAT")
xyz_to_pdb(chlor_xyz, chlor_pdb_tpl, resname="ORG")
xyz_to_pdb(t3_xyz, t3_pdb_tpl, resname="T3H")

inp_iface = (
    f"tolerance 2.0\n"
    f"filetype pdb\n"
    f"output {iface_pdb.resolve()}\n\n"
    f"structure {water_pdb_tpl.resolve()}\n"
    f"  number 1019\n"
    f"  inside box -20. 0. 0. 0. 39. 39.\n"
    f"end structure\n\n"
    f"structure {chlor_pdb_tpl.resolve()}\n"
    f"  number 199\n"
    f"  inside box 0. 0. 0. 21. 39. 39.\n"
    f"end structure\n\n"
    f"structure {t3_pdb_tpl.resolve()}\n"
    f"  centerofmass\n"
    f"  fixed 0. 20. 20. 1.57 1.57 1.57\n"
    f"end structure\n"
)

print("Notebook interface input:\n")
print(inp_iface)
_ = run_packmol(inp_iface, iface_inp)

if iface_pdb.exists():
    print("Wrote:", iface_pdb)
    print("PDB atoms:", count_pdb_atoms(iface_pdb))


In [ ]:
# Visualize interface output (PDB)
if not HAS_3DMOL:
    print("py3Dmol not installed.")
elif not iface_pdb.exists():
    print("Run previous cell first.")
else:
    view = py3Dmol.view()
    view.addModel(open(iface_pdb, "r").read(), "pdb")
    view.setBackgroundColor("white")
    view.setStyle({"resn": "WAT"}, {"licorice": {"radius": 0.12, "colorscheme": "Jmol"}})
    view.addStyle({"elem": "Cl"}, {"sphere": {"radius": 0.52, "color": "green"}})
    view.addStyle({"elem": "C"}, {"stick": {"radius": 0.13}})
    view.addStyle({"elem": "I"}, {"sphere": {"radius": 0.65, "color": "purple"}})
    view.zoomTo()
    view.show()


## 6C) 4246-atom protein, spherical water + ions


In [ ]:
protein_template = examples_dir / "protein.pdb"
print("Protein atom count:", count_pdb_atoms(protein_template))
print("Official solvprotein.inp:\n")
print((examples_dir / "solvprotein.inp").read_text())

solv_inp = outdir / "09_official_solvprotein.inp"
solv_out = outdir / "09_official_solvprotein.pdb"

inp_solv = (
    f"tolerance 2.0\n"
    f"filetype pdb\n"
    f"output {solv_out.resolve()}\n\n"
    f"structure {protein_template.resolve()}\n"
    f"  number 1\n"
    f"  fixed 0. 0. 0. 0. 0. 0.\n"
    f"  centerofmass\n"
    f"end structure\n\n"
    f"structure {(examples_dir / 'water.pdb').resolve()}\n"
    f"  number 16500\n"
    f"  inside sphere 0. 0. 0. 50.\n"
    f"end structure\n\n"
    f"structure {(examples_dir / 'CLA.pdb').resolve()}\n"
    f"  number 20\n"
    f"  inside sphere 0. 0. 0. 50.\n"
    f"end structure\n\n"
    f"structure {(examples_dir / 'SOD.pdb').resolve()}\n"
    f"  number 30\n"
    f"  inside sphere 0. 0. 0. 50.\n"
    f"end structure\n"
)

print("Notebook solvprotein input:\n")
print(inp_solv)
_ = run_packmol(inp_solv, solv_inp)

if solv_out.exists():
    topology_summary(app.PDBFile(str(solv_out)).topology, "Solvprotein output")
    print("Wrote:", solv_out)


In [ ]:
# Visualize solvprotein output
if not HAS_3DMOL:
    print("py3Dmol not installed.")
elif not solv_out.exists():
    print("Run previous cell first.")
else:
    view = py3Dmol.view()
    view.addModel(open(solv_out, "r").read(), "pdb")
    view.setBackgroundColor("white")
    view.setStyle({"chain": "A"}, {"cartoon": {"color": "purple"}})
    view.setStyle({"resn": "HOH"}, {"licorice": {"radius": 0.10, "colorscheme": "Jmol"}})
    view.addStyle({"resn": "CLA"}, {"sphere": {"radius": 0.45, "color": "green"}})
    view.addStyle({"resn": "SOD"}, {"sphere": {"radius": 0.45, "color": "blue"}})
    view.zoomTo()
    view.show()


## 6D) Fun example: write `HELLO` with water molecules

Uses official `water.pdb` as the only structure template.


In [ ]:
PIX = {
    "H": ["#   #", "#   #", "#####", "#   #", "#   #"],
    "E": ["#####", "#    ", "#####", "#    ", "#####"],
    "L": ["#    ", "#    ", "#    ", "#    ", "#####"],
    "O": ["#####", "#   #", "#   #", "#   #", "#####"],
}


def text_points(text: str, pitch=2.5, gap=3.5):
    pts = []
    x0 = 0.0
    for ch in text:
        glyph = PIX.get(ch.upper())
        if glyph is None:
            x0 += gap
            continue
        h = len(glyph)
        w = max(len(row) for row in glyph)
        for iy, row in enumerate(glyph):
            for ix, mark in enumerate(row):
                if mark == "#":
                    pts.append((x0 + ix * pitch, (h - 1 - iy) * pitch))
        x0 += w * pitch + gap
    return np.array(pts, dtype=float)


word = "HELLO"
pts = text_points(word)
pts -= pts.mean(axis=0)
pts += np.array([45.0, 24.0])

z0 = 20.0
jitter = 0.8

hello_inp = outdir / "10_hello_water.inp"
hello_out = outdir / "10_hello_water.pdb"
water_tpl = (examples_dir / "water.pdb").resolve()

blocks = []
for x, y in pts:
    x1, x2 = x - jitter, x + jitter
    y1, y2 = y - jitter, y + jitter
    z1, z2 = z0 - jitter, z0 + jitter
    blocks.append(
        f"structure {water_tpl}\n"
        f"  number 1\n"
        f"  inside box {x1:.3f} {y1:.3f} {z1:.3f} {x2:.3f} {y2:.3f} {z2:.3f}\n"
        f"end structure"
    )

inp_hello = "\n\n".join([
    "tolerance 1.8",
    "filetype pdb",
    f"output {hello_out.resolve()}",
    "seed 20260303",
    *blocks,
])

print("HELLO input preview (first 60 lines):")
for i, line in enumerate(inp_hello.splitlines()[:60], start=1):
    print(f"{i:02d}: {line}")

_ = run_packmol(inp_hello, hello_inp)

if hello_out.exists():
    topology_summary(app.PDBFile(str(hello_out)).topology, f"Water text: {word}")
    print("Wrote:", hello_out)


In [ ]:
# Visualize HELLO water art
if not HAS_3DMOL:
    print("py3Dmol not installed.")
elif not hello_out.exists():
    print("Run previous cell first.")
else:
    view = py3Dmol.view()
    view.addModel(open(hello_out, "r").read(), "pdb")
    view.setBackgroundColor("white")
    view.setStyle({"resn": "HOH"}, {"licorice": {"radius": 0.14, "colorscheme": "Jmol"}})
    view.zoomTo()
    view.show()


## 7) Optional peptides (`warp-pep` + `warp-pack`)

Two examples. Each build is followed by a py3Dmol cell.


In [ ]:
has_warp_pep = shutil.which("warp-pep") is not None
has_warp_pack = shutil.which("warp-pack") is not None
print("warp-pep found:", has_warp_pep)
print("warp-pack found:", has_warp_pack)

pep1 = outdir / "11_pep_alpha.pdb"
packed1 = outdir / "11_pep_alpha_pack.pdb"
cfg1 = outdir / "11_pep_alpha_pack.json"

if has_warp_pep and has_warp_pack:
    subprocess.check_call([
        "warp-pep", "build",
        "--sequence", "AEAAAKEAAAKA",
        "--preset", "alpha-helix",
        "--oxt",
        "--output", str(pep1),
    ])

    cfg_obj1 = {
        "box": {"size": [90.0, 90.0, 90.0]},
        "structures": [{"path": str(pep1), "count": 24}],
    }
    cfg1.write_text(json.dumps(cfg_obj1, indent=2))

    subprocess.check_call([
        "warp-pack",
        "--config", str(cfg1),
        "--output", str(packed1),
        "--format", "pdb",
    ])

    print("Example 1 complete")
    print(" -", pep1)
    print(" -", packed1)
else:
    print("Skipping example 1. Install warp-pep and warp-pack.")


In [ ]:
# Visualize peptide example 1
if not HAS_3DMOL:
    print("py3Dmol not installed.")
elif not packed1.exists():
    print("Run previous cell first.")
else:
    view = py3Dmol.view()
    view.addModel(open(packed1, "r").read(), "pdb")
    view.setBackgroundColor("white")
    view.setStyle({"cartoon": {"color": "purple"}})
    view.zoomTo()
    view.show()


In [ ]:
pep_a = outdir / "12_pep_a.pdb"
pep_b = outdir / "12_pep_b.pdb"
packed2 = outdir / "12_two_peptides_pack.pdb"
cfg2 = outdir / "12_two_peptides_pack.json"

if has_warp_pep and has_warp_pack:
    subprocess.check_call([
        "warp-pep", "build",
        "--three-letter", "ALA-LEU-ALA-LEU-ALA-LEU-GLY-GLY",
        "--output", str(pep_a),
    ])
    subprocess.check_call([
        "warp-pep", "build",
        "--sequence", "RRGGEEKK",
        "--preset", "extended",
        "--output", str(pep_b),
    ])

    cfg_obj2 = {
        "box": {"size": [95.0, 95.0, 95.0]},
        "structures": [
            {"path": str(pep_a), "count": 18},
            {"path": str(pep_b), "count": 18},
        ],
    }
    cfg2.write_text(json.dumps(cfg_obj2, indent=2))

    subprocess.check_call([
        "warp-pack",
        "--config", str(cfg2),
        "--output", str(packed2),
        "--format", "pdb",
    ])

    print("Example 2 complete")
    print(" -", pep_a)
    print(" -", pep_b)
    print(" -", packed2)
else:
    print("Skipping example 2. Install warp-pep and warp-pack.")


In [ ]:
# Visualize peptide example 2
if not HAS_3DMOL:
    print("py3Dmol not installed.")
elif not packed2.exists():
    print("Run previous cell first.")
else:
    view = py3Dmol.view()
    view.addModel(open(packed2, "r").read(), "pdb")
    view.setBackgroundColor("white")
    view.setStyle({"cartoon": {"colorscheme": "yellowCarbon"}})
    view.zoomTo()
    view.show()


## Mini missions

1. Change `n_side` (DIY + PACKMOL water) and compare the structures.
2. In mixture, try water/urea counts of your choice.
3. In interface, move the hormone fixed position and re-visualize.
4. Replace `HELLO` with your own text.
5. Optional: compare packed peptide layouts from example 1 and 2.
